# comparison

In [ ]:
from pathlib import Path

from combra import angles
from combra.metrics import compare_folders, find_kimg_parquets

GRAIN_LABEL = {'Ultra_Co11': 'small grain',
               'Ultra_Co25': 'medium grain',
               'Ultra_Co6_2': 'large grain'}

GRID = Path('./data/angles')
H5 = GRID.parent / 'h5'
MIN_SEG_LEN = 5.0
RUN = '00017-diffit-256-gpus2-batch192'
ETHALON = 'o_bc_left_4x_1536_1024x1024_256x256_rgb_N360'

reference_path = angles.output_directory(GRID, ETHALON, MIN_SEG_LEN) / 'angles_n360.parquet'
# N10000 reference row first, then every kimg snapshot chronologically.
folder_paths = find_kimg_parquets(GRID, RUN, n=1000, msl=MIN_SEG_LEN, final_tag='N10000')
# Each checkpoint's gen h5 shares its parquet folder's stem.
gen_h5_map = {str(p): str(H5 / f"{p.parent.name.replace(f'_msl{int(MIN_SEG_LEN)}', '')}.h5")
              for p in folder_paths}

# No class map anywhere: combra pairs h5 groups and parquet rows by grain-class
# name, which every artifact records. scale=1 prints wdist in raw degrees; gen_n
# caps gen images per class so the N10000 reference row does not load all 10k
# images for the image metrics.
records = compare_folders(folder_paths, str(reference_path),
                          steps=[2], scale=1, verbose=True,
                          real_h5=str(H5 / f'{ETHALON}.h5'), gen_h5_map=gen_h5_map,
                          image_metrics=True, gen_n=1000)

In [ ]:
from combra.metrics import plot_metrics_overlay

# All metrics as |value| on one log axis vs kimg, one figure per class. The
# N10000 reference row carries a non-kimg tag, so drop it from the curves.
ckpt_records = [r for r in records if r['kimg'].split('_')[0].isdigit()]

for cls, grain in GRAIN_LABEL.items():
    fig = plot_metrics_overlay(ckpt_records, cls,
                               title=f'{grain} ({cls}) — diffit 256×256',
                               save_path=f'metrics_overlay_{cls}_step2.png')
    fig.show()

In [ ]:
import pandas as pd

from combra.metrics import all_metrics_by_sample_size

# Full metric suite over an N sweep, per (resolution, generator) -> one tidy
# parquet. Heavy: runs Inception / DINOv2 per N.
SOURCES = {
    256: {'real':   H5 / f'{ETHALON}.h5',
          'san':    H5 / 'gen_san_256x256_N100_000.h5',
          'diffit': H5 / '00017-diffit-256-gpus2-batch192_N10000.h5'},
    512: {'real':   H5 / 'o_bc_left_4x_1536_1024x1024_512x512_rgb_N360.h5',
          'san':    H5 / 'gen_san_512x512_N100_000.h5',
          'diffit': H5 / '00018-diffit-512-gpus4-batch256_N10000.h5'},
}
N_SWEEP = [100, 250, 1000, 10000]

rows = []
for res, group in SOURCES.items():
    for kind in ('san', 'diffit'):
        recs = all_metrics_by_sample_size(str(group['real']), str(group[kind]),
                                        ns=N_SWEEP, step=2.0)
        rows.extend({'kind': kind, 'resolution': res, **r} for r in recs)

metrics_all = pd.DataFrame.from_records(rows)
metrics_all.to_parquet('all_metrics_vs_n_step2.parquet', index=False)
print(f'wrote {len(metrics_all)} rows -> all_metrics_vs_n_step2.parquet')
metrics_all.head()

# training-curve grid (from tensorboards)

Per-resolution **3×2 grid** straight from the `tensorboards/` event files (one per
model×resolution). Columns are the two models — **san** (left) and **diffit**
(right); rows are the metric families:

1. **physical metrics** — angle W-dist (`w1/w2`, `circular_w1/w2`) and the
   Gaussian-fit relative errors (`mu`, `sigma`, `amp`);
2. **training metrics** — FID / CMMD / FD-DINOv2 (10k);
3. **losses** — model-specific (`G/D` for san, `train/mse/vb` for diffit).

The two runs live on very different axes (diffit ≈28 000 kimg, san ≈1 600 kimg)
and every metric has its own y-scale, so the x-axis is normalized to training
**progress fraction** (`kimg / max kimg`) and each curve is **min-max
normalized** to [0, 1] — the panels show convergence *shape*, not raw values.

In [ ]:
from pathlib import Path

from combra.io import read_tb_scalars

# One tfevents file per (model, resolution); only the loss tags differ.
TB = Path('tensorboards')
RUNS = {
    256:  {'diffit': 'diffit-256x256',   'san': 'san-256x256'},
    512:  {'diffit': 'diffit-512-512',   'san': 'san-512-512'},
    1024: {'diffit': 'diffit-1024x1024', 'san': 'san-1024-1024'},
}

tb_data = {res: {m: read_tb_scalars(TB / fname) for m, fname in models.items()}
           for res, models in RUNS.items()}
print({res: {m: len(sc) for m, sc in d.items()} for res, d in tb_data.items()})

In [ ]:
import numpy as np

from combra.metrics import plot_training_curve_grid

KIMG_TAG = {'diffit': 'Train/kimg', 'san': 'Progress/kimg'}
# Grid rows: (family label, tags, scalar-name prefix).
ROWS = [
    ('physical metrics',
     ['combra_w2', 'combra_mu1', 'combra_mu2',
      'combra_sigma1', 'combra_sigma2', 'combra_share1', 'combra_share2'], 'Metrics/'),
    ('training metrics (FID / CMMD / FD-DINOv2)',
     ['combra_fid10k', 'combra_cmmd10k', 'combra_fd_dinov2_10k'], 'Metrics/'),
    ('losses',
     {'diffit': ['Train/Loss/train', 'Train/Loss/mse', 'Train/Loss/vb'],
      'san': ['Loss/G/loss', 'Loss/D/loss']}, ''),
]

# SAN 512x512 collapsed twice: a transient burst mid-training and a terminal
# divergence. For those panels only, drop the tail and (on the metric panels,
# not the losses) bridge the burst with a dashed segment.
SAN512_CUT_CENTER = (0.42, 0.50)
SAN512_CUT_TAIL = 0.89


def san512_override(model, res, tag, x, y):
    if not (model == 'san' and res == 512):
        return None
    keep = x < SAN512_CUT_TAIL
    x, y = x[keep], y[keep].copy()
    if 'Loss' in tag:
        return x, y, None
    lo, hi = SAN512_CUT_CENTER
    win = (x >= lo) & (x <= hi)
    left, right = np.where(x < lo)[0], np.where(x > hi)[0]
    if win.any() and len(left) and len(right):
        li, ri = left[-1], right[0]        # real points bracketing the window
        y[win] = np.interp(x[win], [x[li], x[ri]], [y[li], y[ri]])
        return x, y, win
    return x, y, None


for res in tb_data:
    fig = plot_training_curve_grid(tb_data[res], ROWS, ['san', 'diffit'], KIMG_TAG,
                                   resolution=res, overrides=san512_override,
                                   save_path=f'tb_grid_{res}.png')
    fig.show()